# 10. More training data (Colab GPU)

**Why.** The final GRU scores 0.891 on cross-validation but only 0.589 on the leaderboard. The EDA found that names the model never saw in training are hard: Ukrainian unseen names are recalled 24-25% of the time, seen names 76-87%. Notebook 04 found that more data was the biggest lever (0.856 -> 0.885 from 10k to 100k rows per language, still rising), and the datasets hold far more rows than we use (960k Ukrainian and 6.4M Russian `train` rows).

**What changes.** Only the amount of data: 300,000 rows per language instead of 100,000. Everything else is the final recipe of notebook 06 (GRU + CRF, frozen fastText, seed, optimizer, decoding), so the difference in score comes from the data alone. The cross-validation folds and the held-out rows are the same as in notebook 06, so the CV scores are directly comparable.

**Inputs.** The run needs no raw CSV files: `ner_bundle.zip` holds the code, the competition test files and a row cache built locally with `build_cache(extra_per_source=900_000)` from the frozen manifests (the same 63,388 CV rows, plus up to 900,000 `train` rows per language outside CV, so `ROWS` can be raised without a new upload).

**How to run on Colab.** Upload `ner_bundle.zip` to `MyDrive/ai301_more_data/`, choose a GPU runtime with high RAM, and run all cells. The three folds train in parallel on one GPU. Results are written to Drive, so a disconnected session continues where it stopped.

In [2]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ai301_more_data")
    ROOT = Path("/content/ai301")
    if not (ROOT / "eda").exists():
        with zipfile.ZipFile(PROJECT / "ner_bundle.zip") as bundle:
            bundle.extractall(ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightning", "pytorch-crf", "tokenizers"],
                   check=True)
    os.environ["NER_WORK_DIR"] = str(PROJECT / "work")
    cache = PROJECT / "work" / "cache"
    cache.mkdir(parents=True, exist_ok=True)
    for name in ("rows.parquet", "vectors_aligned_200000.npz"):
        if not (cache / name).exists():
            # Drive is another device, so the file is copied; the temporary name guards against a cut copy
            shutil.copy(ROOT / "cache" / name, cache / f"{name}.part")
            (cache / f"{name}.part").replace(cache / name)
else:
    ROOT = Path.cwd()
    while not (ROOT / "eda").exists():
        ROOT = ROOT.parent
    os.environ.setdefault("NER_WORK_DIR", str(ROOT / "data" / "ner_work" / "more_data"))
os.environ["NER_DATA_DIR"] = str(ROOT / "data" / "kse-ua-location-extraction-2026")

import torch

os.environ.setdefault("NER_WANDB", "off")
os.environ["NER_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
print("Device:", os.environ["NER_DEVICE"], "| work folder:", os.environ["NER_WORK_DIR"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | work folder: /content/drive/MyDrive/ai301_more_data/work


In [3]:
import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, majority_vote, row_key, write_submission
from nerkit.config import ARTIFACTS_DIR, RESULTS_DIR, WORK_DIR
from nerkit.data import build_cache, load_labeling_sample, load_test
from nerkit.decisions import DECODING, base_config
from nerkit.experiment import run_experiment
from nerkit.metrics import evaluate

NOTEBOOK = "10_more_data"
FOLDS = (0, 1, 2)
ROWS = 300000  # per language; notebook 06 used 100,000
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}


def more_data_config(fold):
    return base_config(fold=fold, uk_rows=ROWS, ru_rows=ROWS)


build_cache()  # the bundled cache is used; this only checks that it is in place
print("Runs:", {fold: more_data_config(fold).key() for fold in FOLDS})

Runs: {0: '00b6486099dd', 1: 'a503b30fda5a', 2: 'b1b13357bc6f'}


## Train the three folds in parallel

Each fold runs in its own process; the model is small, so one GPU holds all three. A finished fold is skipped on rerun. Progress: open `MyDrive/ai301_more_data/work/logs/fold0.log` in the Colab file browser; it gets one line per epoch (validation F1 and seconds).

In [4]:
TRAIN = '''
import sys
sys.path[:0] = {paths!r}
from nerkit.decisions import base_config
from nerkit.engine import TaggerModule
from nerkit.experiment import run_experiment

epoch_end = TaggerModule.on_validation_epoch_end


def epoch_end_with_report(self):  # training is silent otherwise; these lines are the progress log
    epoch_end(self)
    if self.history:
        last = self.history[-1]
        print(f"epoch {{last['epoch']}}: val_f1 {{last['val_f1']:.3f}}, {{last['seconds']:.0f}} s", flush=True)


TaggerModule.on_validation_epoch_end = epoch_end_with_report
run_experiment("more_data_fold{fold}", base_config(fold={fold}, uk_rows={rows}, ru_rows={rows}{extra}), "{notebook}")
'''
logs = WORK_DIR / "logs"
logs.mkdir(parents=True, exist_ok=True)
processes = {}
for fold in FOLDS:
    script = TRAIN.format(paths=[str(ROOT), str(ROOT / "model")], fold=fold, rows=ROWS, notebook=NOTEBOOK,
                          extra="")
    with (logs / f"fold{fold}.log").open("w") as log:
        processes[fold] = subprocess.Popen([sys.executable, "-c", script], stdout=log, stderr=subprocess.STDOUT)
for fold, process in processes.items():
    if process.wait() != 0:
        raise RuntimeError(f"fold {fold} failed: " + (logs / f"fold{fold}.log").read_text()[-2000:])
records = {fold: run_experiment(f"more_data_fold{fold}", more_data_config(fold), NOTEBOOK) for fold in FOLDS}

more_data_fold0: already finished, F1 mix 0.893
more_data_fold1: already finished, F1 mix 0.901
more_data_fold2: already finished, F1 mix 0.901


**Training was stopped by hand.** The run was started with the notebook 06 limit of 10 epochs; one epoch over 600,000 rows took 23-28 minutes on an L4 GPU with the three folds sharing it, so 10 epochs did not fit the deadline. Each process was stopped by setting Lightning's `trainer.should_stop` from outside (with `gdb`); training then ends after the current batch, the best epoch on the inner validation split is restored, and prediction and saving run as usual. Fold 2 stopped during epoch 3 (best: the partial epoch 3, inner F1 0.902), fold 1 during epoch 6 (best epoch 4, 0.915), fold 0 during epoch 6 (best epoch 5, 0.910). The `epochs` column counts the epochs that were validated.

## Cross-validation against notebook 06

Same folds, same held-out rows; only the amount of training data differs.

In [5]:
# notebook 06 kept its per-fold results in final_cv/summary.json
saved = json.loads((ROOT / "data" / "ner_work" / "final_cv" / "summary.json").read_text(encoding="utf-8"))
before = {fold: saved["folds"][f"fold{fold}"] for fold in FOLDS}


def summary(scores, training_rows, epochs, minutes):
    return dict(heldout_f1=scores["heldout"]["mix"]["f1"], precision=scores["heldout"]["mix"]["precision"],
                recall=scores["heldout"]["mix"]["recall"], posts_f1=scores["posts"]["mix"]["f1"],
                uk_unseen_recall=scores["unseen_recall"].get("uk"), labeling_f1=scores["labeling"]["pooled"]["f1"],
                training_rows=training_rows, epochs=epochs, minutes=minutes)


rows = []
for fold in FOLDS:
    old = before[fold]
    rows.append(dict(fold=fold, data="100k (nb 06)", **summary(old["scores"], old["training_rows"], old["epochs"],
                                                              old["minutes"])))
    new = records[fold]
    rows.append(dict(fold=fold, data="300k", **summary(new["scores"], new["training_rows"], len(new["history"]),
                                                      new["total_seconds"] / 60)))
table = pd.DataFrame(rows)
display(table.set_index(["fold", "data"]).round(3))
display(table.groupby("data")[["heldout_f1", "posts_f1", "uk_unseen_recall", "labeling_f1"]].agg(["mean", "std"]).round(3))

heldout_f1  precision  recall  posts_f1  uk_unseen_recall  \
fold data                                                                      
0    100k (nb 06)       0.885      0.914   0.858     0.863             0.600   
     300k               0.893      0.910   0.877     0.875             0.588   
1    100k (nb 06)       0.896      0.912   0.880     0.873             0.641   
     300k               0.901      0.929   0.875     0.885             0.561   
2    100k (nb 06)       0.894      0.906   0.881     0.882             0.661   
     300k               0.901      0.890   0.912     0.888             0.667   

                   labeling_f1  training_rows  epochs  minutes  
fold data                                                       
0    100k (nb 06)        0.606         199606       6   39.079  
     300k                0.657         598816       6  131.103  
1    100k (nb 06)        0.722         199602      10   62.448  
     300k                0.667         598812       6  131.272  
2    100k (nb 06)        0.676         199616       6   10.514  
     300k                0.684         598826       3   66.393

heldout_f1        posts_f1        uk_unseen_recall         \
                   mean    std     mean    std             mean    std   
data                                                                     
100k (nb 06)      0.891  0.006    0.873  0.009            0.634  0.031   
300k              0.899  0.005    0.882  0.007            0.605  0.055   

             labeling_f1         
                    mean    std  
data                             
100k (nb 06)       0.668  0.058  
300k               0.669  0.013

## Test predictions

The same two combinations as notebook 07: the vote (a string is kept when 2 of the 3 fold models predict it; the best leaderboard score so far, 0.58914) and the average of log-probabilities and CRF transitions.

In [6]:
test_rows, labeling_rows = load_test(), load_labeling_sample()
keys = [records[fold]["key"] for fold in FOLDS]


def vote(rows, split):
    single = []
    for key in keys:
        logits, meta = load_logits(key, split)
        single.append(decode_split(logits, rows, crf=meta["crf"], **DECODE))
    return majority_vote(single, 2)


def average(rows, split):
    loaded = [load_logits(key, split) for key in keys]
    logits, crf = combine_models([l for l, _ in loaded], [m for _, m in loaded], {row_key(r): r["text"] for r in rows})
    return decode_split(logits, rows, crf=crf, **DECODE)


overview = []
for name, method, file in (("vote, 2 of 3", vote, "more_data_vote2.csv"), ("average", average, "more_data_average.csv")):
    predictions = method(test_rows, "test")
    write_submission(WORK_DIR / "submissions" / file, test_rows, predictions)
    labeling = evaluate(labeling_rows, method(labeling_rows, "labeling"))["pooled"]
    overview.append(dict(method=name, file=file, posts_with_locations=sum(bool(p) for p in predictions),
                         locations=sum(len(p) for p in predictions), labeling_f1=labeling["f1"]))
display(pd.DataFrame(overview).set_index("method").round(3))

,file,posts_with_locations,locations,labeling_f1
method,,,,
"vote, 2 of 3",more_data_vote2.csv,325,748,0.703
average,more_data_average.csv,327,759,0.676


## Save the results

`more_data_results.zip` (on Drive, next to the bundle) holds the result records, the saved logits of the three runs and the two submission files. The model weights are left out to keep it small.

In [7]:
archive = WORK_DIR / "more_data_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as package:
    package.write(RESULTS_DIR / f"{NOTEBOOK}.jsonl", f"results/{NOTEBOOK}.jsonl")
    for key in keys:
        for path in (ARTIFACTS_DIR / key).iterdir():
            if path.name != "model.pt":
                package.write(path, f"artifacts/{key}/{path.name}")
    for file in ("more_data_vote2.csv", "more_data_average.csv"):
        package.write(WORK_DIR / "submissions" / file, f"submissions/{file}")
print(archive, f"{archive.stat().st_size / 1e6:.0f} MB")

/content/drive/MyDrive/ai301_more_data/work/more_data_results.zip 22 MB


## Leaderboard results

In [8]:
LEADERBOARD = {  # public F1, copied from Kaggle
    "100k rows, vote 2 of 3 (notebook 07)": 0.58914,
    "100k rows, average (notebook 07)": 0.58787,
    "300k rows, vote 2 of 3": 0.5817,
    "300k rows, average": 0.57313,
}
display(pd.Series(LEADERBOARD, name="public_f1").to_frame())

                                      public_f1
100k rows, vote 2 of 3 (notebook 07)    0.58914
100k rows, average (notebook 07)        0.58787
300k rows, vote 2 of 3                  0.58170
300k rows, average                      0.57313

**Conclusion.** More data improves every local measure (CV 0.899 against 0.891, synthetic posts 0.882 against 0.873) but lowers the leaderboard (0.582 and 0.573 against 0.589 and 0.588). The extra rows are more source sentences of the same kind, so the model fits the source domain better without learning more about Telegram posts. The Ukrainian unseen-name recall is not comparable between the two runs: with three times more data, fewer names are unseen, and the remaining ones are harder.